In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

SILVER = "review_silver"
GOLD = "fact_review"

df_review = spark.read.table(SILVER)

df_fact = (
    df_review
    .withColumn("date_id", F.date_format("date", "yyyyMMdd").cast("int"))
    .select(
        "review_id",
        "business_id",
        "user_id",
        "date_id",
        "stars",
        "useful",
        "funny",
        "cool",
        "_ingest_ts"
    )
)

if spark.catalog.tableExists(GOLD):
    delta = DeltaTable.forName(spark,GOLD)

    (
        delta.alias("t")
        .merge(
            df_fact.alias("s"),
            "t.review_id = s.review_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        df_fact.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD)
    )

StatementMeta(, 0f2bbf63-f8aa-425a-83b1-e56c28145359, 3, Finished, Available, Finished, False)